# pyannote (speaker diarization)

**Domain:** Speech & Audio  ·  *recommended addition*  ·  **runnable:** yes

A refresher on **speaker diarization** — answering *"who spoke when?"* — and on
[`pyannote.audio`](https://github.com/pyannote/pyannote-audio), the de-facto open-source
toolkit. We build the embed → cluster pipeline from scratch on tiny synthetic data, implement the
DER metric, and show the real `pyannote` call shape gated behind a Hugging Face token.

## 1. What & Why

**Speaker diarization** partitions an audio stream into homogeneous segments and labels each with
an *anonymous* speaker tag (`SPEAKER_00`, `SPEAKER_01`, …). It answers **"who spoke when"** — not
*what* was said (that's ASR) and not *who exactly* the person is (that's speaker identification /
verification). The speakers are discovered, unlabeled clusters; tag `SPEAKER_00` in one file has
no relation to `SPEAKER_00` in another.

**The problem it solves.** Raw meeting/call/interview audio is a single waveform with several
people talking, often over each other. Downstream tasks need structure:

- **Meeting & call transcripts** — attribute each sentence to a speaker (combine with ASR).
- **Conversational analytics** — talk-time per participant, turn-taking, interruptions.
- **Dataset preparation** — split multi-speaker recordings into per-speaker chunks for TTS/ASR
  training.
- **Compliance / medical / legal** — separate doctor vs. patient, agent vs. customer.

**Reach for it** whenever you have multi-speaker audio and need per-speaker structure. **Skip it**
when you already have separate channels per speaker (telephony often gives you stereo with one
speaker per channel — just process each channel) or when a single dominant speaker means a plain
VAD is enough.

## 2. Mental Model

Think of diarization as **clustering short voice fingerprints along a timeline**:

```
  waveform ──► [ VAD ] ──► speech regions only (drop silence/noise)
                  │
                  ▼
            [ segmentation ] ─► fine sliding windows, incl. overlap detection
                  │
                  ▼
            [ embedding ] ───► each window → a d-vector (a point in voice-space)
                  │              same voice  ⇒ nearby points
                  ▼
            [ clustering ] ──► group points → SPEAKER_00, SPEAKER_01, …
                  │
                  ▼
            timeline:  0–3s SPEAKER_00 · 3–6s SPEAKER_01 · 6–8s SPEAKER_00
```

The crux is the **embedding + clustering** stage: a neural net maps each little window of speech to
a vector such that *the same voice lands in the same region of space*. Then any clustering
algorithm (agglomerative, spectral) groups those vectors. The labels are arbitrary cluster ids —
diarization is **unsupervised at inference time**. Modern `pyannote` folds VAD, overlap-aware
segmentation, embedding, and clustering into one trained *pipeline*, but the mental model above is
exactly what's happening inside.

## 3. Key Concepts

- **Diarization vs. recognition vs. verification.** Diarization = *how many speakers and when*
  (anonymous). Recognition/identification = *which known person*. Verification = *are these two
  clips the same person*. Diarization needs none of the speakers' identities.
- **VAD (Voice Activity Detection).** Binary speech/non-speech gate; removes silence and noise so
  you only embed actual speech.
- **Speaker embedding (d-vector / x-vector).** A fixed-length vector summarizing *who* is speaking
  in a window, trained so same-speaker vectors are close (cosine). pyannote uses a model derived
  from ECAPA-TDNN / SincNet.
- **Segmentation & overlap.** Splitting speech into short turns; pyannote's segmentation model is
  *overlap-aware*, predicting when two people talk simultaneously.
- **Clustering.** Grouping embeddings into speakers — agglomerative (with a distance threshold)
  when the speaker count is unknown, or constrained when you pass `num_speakers`.
- **RTTM.** The standard output format (NIST Rich Transcription Time Marked): one line per
  segment with start, duration, and speaker label.
- **DER (Diarization Error Rate).** The headline metric = (false alarm + missed speech + speaker
  confusion) / total reference speech time. Lower is better; computed after an optimal
  reference↔hypothesis speaker mapping. A *collar* around boundaries is often allowed.

## 4. Setup

The real pipeline needs PyTorch and the gated `pyannote` models; the two from-scratch examples
below need only NumPy + scikit-learn + SciPy (CPU, no downloads).

```bash
pip install pyannote.audio          # pulls torch, torchaudio, etc. (~GB)
pip install numpy scikit-learn scipy  # enough for the worked examples here
```

**Gated models.** `pyannote/speaker-diarization-3.1` is hosted on Hugging Face behind a
user-agreement gate. To run it you must (1) create a free HF account, (2) accept the conditions on
the model page *and* its dependency `pyannote/segmentation-3.0`, and (3) pass an access token. We
read that token from the `HF_TOKEN` env var so the notebook still executes without it.

In [ ]:
import sys
print("Python", sys.version.split()[0])
for mod in ("numpy", "sklearn", "scipy"):
    try:
        m = __import__(mod)
        print(f"{mod:8} {m.__version__}")
    except ImportError:
        print(f"{mod:8} MISSING — pip install {mod}")

# The heavy, gated pyannote model is optional; we check for it but never require it.
try:
    import pyannote.audio  # noqa: F401
    print("pyannote.audio available (real-pipeline cell can run if HF_TOKEN is set)")
except ImportError:
    print("pyannote.audio not installed — real-pipeline cell will show call shape only")

## 5. Worked Examples

### Example 1 — The embed → cluster core, from scratch

This is the heart of every diarization system. We skip the neural embedder and *fabricate*
2-D speaker embeddings for eight 2-second windows drawn from two speakers, then recover the
speakers with agglomerative clustering on cosine distance — exactly what pyannote does after its
embedding model, just with real d-vectors in ~192 dimensions.

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

rng = np.random.default_rng(0)
# Two speakers occupy two regions of "voice space"; real embeddings are ~192-D.
centers = {"Alice": np.array([1.0, 0.0]), "Bob": np.array([-1.0, 0.5])}
truth   = ["Alice", "Alice", "Bob", "Alice", "Bob", "Bob", "Alice", "Bob"]
starts  = np.arange(len(truth)) * 2.0  # 2-second windows

# One noisy embedding per window, then L2-normalize (cosine clustering wants unit vectors).
embeds = np.stack([centers[s] + rng.normal(0, 0.25, size=2) for s in truth])
embeds /= np.linalg.norm(embeds, axis=1, keepdims=True)

# Unknown speaker count in the wild -> threshold; here we know there are 2.
labels = AgglomerativeClustering(
    n_clusters=2, metric="cosine", linkage="average"
).fit_predict(embeds)

print("window  start-end     truth    cluster")
for s, t, l in zip(starts, truth, labels):
    print(f"  {int(s/2):2}   {s:4.0f}-{s+2:4.0f}s   {t:7}  SPEAKER_{l:02d}")
print("\nClusters perfectly recover the two speakers (label ids are arbitrary).")

The clustering recovers the two speakers exactly — note the cluster *ids* are arbitrary
(`SPEAKER_00` might be Bob). That label arbitrariness is precisely why scoring diarization needs an
optimal mapping step, which is the next example.

### Example 2 — Diarization Error Rate (DER) with optimal speaker mapping

DER compares a hypothesis timeline against a reference. Because speaker labels are arbitrary, you
first find the best reference↔hypothesis label assignment (Hungarian algorithm), then count
mismatched speech time. We frame the timelines at 100 ms resolution and compute DER directly —
this is a simplified version of what `pyannote.metrics` does (no collar, no overlap).

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def to_frames(segments, step=0.1, end=8.0):
    """(speaker, start, end) segments -> one speaker label per `step`-second frame."""
    frames = [None] * int(round(end / step))
    for spk, s, e in segments:
        for k in range(int(round(s / step)), int(round(e / step))):
            if 0 <= k < len(frames):
                frames[k] = spk
    return frames

# Reference (ground truth) vs. hypothesis (a diarizer's guess, slightly off at boundaries).
reference  = [("A", 0, 3), ("B", 3, 6), ("A", 6, 8)]
hypothesis = [("spk1", 0, 2.7), ("spk2", 2.7, 5.5), ("spk1", 5.5, 8)]
ref_f, hyp_f = to_frames(reference), to_frames(hypothesis)

ref_spk = sorted({x for x in ref_f if x})
hyp_spk = sorted({x for x in hyp_f if x})

# Cost matrix = overlap frames; maximize overlap == minimize negative overlap.
overlap = np.zeros((len(ref_spk), len(hyp_spk)))
for r, h in zip(ref_f, hyp_f):
    if r is not None and h is not None:
        overlap[ref_spk.index(r), hyp_spk.index(h)] += 1
rows, cols = linear_sum_assignment(-overlap)
mapping = {hyp_spk[c]: ref_spk[r] for r, c in zip(rows, cols)}

total   = sum(r is not None for r in ref_f)
correct = sum(r is not None and mapping.get(h) == r for r, h in zip(ref_f, hyp_f))
der = (total - correct) / total

print("best label mapping (hyp -> ref):", mapping)
print(f"reference speech frames : {total}")
print(f"correctly labeled frames: {correct}")
print(f"DER = (total - correct)/total = {der:.1%}")

DER ≈ 10% here — all of it **speaker confusion at the boundaries**, where the hypothesis turn
times drift from the reference. In real evaluation you'd also account for *missed speech* and
*false alarm* (speech the diarizer dropped or invented), and usually forgive a small *collar*
(±250 ms) around each boundary since exact turn times are inherently fuzzy.

### Example 3 — The real `pyannote` pipeline (gated behind `HF_TOKEN`)

This is the actual API you'd use in production. It's gated behind an env check so the notebook runs
without the multi-GB model download or a token. The call shape is the part worth memorizing: load a
pretrained pipeline, optionally move it to GPU, call it on a wav file, iterate the RTTM-style
result.

In [ ]:
import os

if os.getenv("HF_TOKEN"):
    from pyannote.audio import Pipeline
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=os.environ["HF_TOKEN"],
    )
    # import torch; pipeline.to(torch.device("cuda"))  # ~real-time on CPU, much faster on GPU

    diarization = pipeline("meeting.wav")  # or num_speakers=2 / min_speakers=/max_speakers=
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        print(f"{turn.start:6.1f}s - {turn.end:6.1f}s  {speaker}")
    # diarization.write_rttm(open("meeting.rttm", "w"))
else:
    print("HF_TOKEN not set — showing the expected output shape instead:")
    print("Accept the user agreement for pyannote/speaker-diarization-3.1 and")
    print("pyannote/segmentation-3.0 on huggingface.co, then set HF_TOKEN.\n")
    print("   0.5s -   3.2s  SPEAKER_00")
    print("   3.2s -   6.0s  SPEAKER_01")
    print("   6.0s -   8.1s  SPEAKER_00")
    print("# itertracks(yield_label=True) yields (Segment, track_id, label)")

To attach **words** to speakers, run ASR separately and intersect the word timestamps with these
turns — that's exactly what [WhisperX](https://github.com/m-bain/whisperX) does (Whisper for
transcription + pyannote for diarization, see the `faster-whisper-whisperx` notebook).

## 6. Gotchas & Pitfalls

- **The gated-model dance.** `from_pretrained` fails with a 401/403 until you accept the license
  on **both** `pyannote/speaker-diarization-3.1` *and* its `pyannote/segmentation-3.0` dependency
  and pass a valid `HF_TOKEN`. This is the #1 first-run error.
- **Unknown speaker count.** By default the pipeline estimates the number of speakers — and it's
  often wrong on short or noisy audio. If you know it, pass `num_speakers=N` (or
  `min_speakers`/`max_speakers`) for a big accuracy jump.
- **Overlapping speech is hard.** When people talk over each other, single-label diarization must
  pick one; overlap is a major DER contributor. pyannote 3.x is overlap-aware but not perfect.
- **Sample rate & channels.** Models expect **16 kHz mono**. Feeding 44.1 kHz stereo silently
  degrades results — resample and downmix first (e.g. with `torchaudio` or `ffmpeg`).
- **Labels aren't stable across files.** `SPEAKER_00` in file A ≠ `SPEAKER_00` in file B. To track
  a person across recordings you need *speaker identification/verification* on top, not diarization
  alone.
- **DER is not accuracy.** A 10% DER doesn't mean "90% of words right." DER is over *speech time*
  and mixes miss + false-alarm + confusion. Always report whether a collar/overlap was used —
  numbers aren't comparable otherwise.
- **Short segments & latency.** Very short turns (back-channels like "uh-huh") are frequently
  missed or merged. The pipeline is also offline/batch — not designed for low-latency streaming.
- **CPU is slow-ish but fine.** Expect roughly real-time on CPU for the 3.1 pipeline; GPU gives a
  large speedup for long files. Don't assume you need a GPU just to try it.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off |
|---|---|---|
| **pyannote.audio** | Open-source, SOTA-ish accuracy, full control, on-prem | Gated HF models, PyTorch dependency, tuning needed |
| **NVIDIA NeMo diarization** | GPU shops already in the NeMo/Riva stack; MSDD overlap handling | Heavier framework, steeper setup |
| **WhisperX** | You want *transcript + speaker labels* in one shot | Wraps pyannote anyway; same gating, less control |
| **Cloud (AWS Transcribe, Azure, Google STT)** | Zero ML ops, diarization as an API flag | Per-minute cost, data leaves premises, less tunable |
| **Per-channel split (no diarization)** | Telephony/stereo with one speaker per channel | Only works when channels are already separated |
| **Plain VAD + your own clustering** | Learning, or a constrained known-2-speaker case | You reimplement what pyannote already does well |

**Rules of thumb.** Need words *and* speakers → **WhisperX**. Need just the diarization with
control / on-prem → **pyannote**. Already on NVIDIA Riva/NeMo → **NeMo**. Want it as a managed API
and cost/privacy is fine → **cloud**. Already have separate channels → **don't diarize at all**,
just process each channel.

## 8. Resources

- **pyannote.audio (GitHub)** — the toolkit, pretrained pipelines, and tutorials:
  https://github.com/pyannote/pyannote-audio
- **`pyannote/speaker-diarization-3.1` model card** — accept the license here before use:
  https://huggingface.co/pyannote/speaker-diarization-3.1
- **`pyannote.metrics`** — DER and friends, done properly (collar, overlap, mapping):
  https://github.com/pyannote/pyannote-metrics
- **WhisperX** — ASR + diarization + word-level alignment in one pipeline:
  https://github.com/m-bain/whisperX
- **NIST RTTM format reference** — the standard diarization output format:
  https://web.archive.org/web/20170119114252/http://www.itl.nist.gov/iad/mig/tests/rt/2009/docs/rt09-meeting-eval-plan-v2.pdf
- **Bredin et al., "pyannote.audio: neural building blocks for speaker diarization" (ICASSP 2020)**:
  https://arxiv.org/abs/1911.01255

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def to_frames(segments, step=0.1, end=8.0):
    ...


def der(reference, hypothesis, step=0.1, end=8.0):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE